# 04 — Image Captioning Results

Phân tích kết quả Task 2 (Flickr8k):
1. BLEU-1..4 comparison (quantum vs classical)
2. Training curves
3. Sinh caption mẫu từ checkpoint đã train
4. Export LaTeX table

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"
CKPT_DIR = PROJECT_ROOT / "checkpoints"
FIG_DIR = PROJECT_ROOT / "paper" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_json(path):
    p = Path(path)
    if not p.exists():
        print(f"[missing] {p}")
        return None
    with open(p) as f:
        return json.load(f)


caption_runs = load_json(RESULTS_DIR / "caption_results.json") or {}
results_all = load_json(RESULTS_DIR / "results.json") or {}
print("Available caption runs:", list(caption_runs.keys()))

## 1. BLEU comparison

In [ ]:
rows = []
for name, r in caption_runs.items():
    rows.append({
        "run": name,
        "best_val_bleu4": r.get("best_val_bleu4"),
        "test_bleu4": (r.get("test") or {}).get("bleu_4"),
        "total_params": (r.get("params") or {}).get("total"),
        "train_hours": ((r.get("train_time_sec") or 0) / 3600),
    })

df = pd.DataFrame(rows).set_index("run").sort_values("test_bleu4", ascending=False)
df.round(4)

In [ ]:
if len(df):
    fig, ax = plt.subplots(figsize=(9, max(3, 0.5 * len(df))))
    colors = ["#2196F3" if "qmmf" in i else "#FF9800" for i in df.index]
    df["test_bleu4"].plot.barh(ax=ax, color=colors)
    ax.set_xlabel("BLEU-4 (test)")
    ax.set_title("Image Captioning: BLEU-4 by Model")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "caption_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Chưa có kết quả — chạy experiments/run_captioning.py trước.")

## 2. Training curves

In [ ]:
plt.figure(figsize=(10, 5))
for name, r in caption_runs.items():
    hist = r.get("history", [])
    if hist:
        plt.plot([h["epoch"] for h in hist], [h["val_bleu4"] for h in hist], label=name)
plt.xlabel("epoch")
plt.ylabel("Val BLEU-4")
plt.title("Validation BLEU-4 during training")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "caption_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Generate captions mẫu từ checkpoint

In [ ]:
# Load model + checkpoint (chỉ chạy được khi đã train)
import torch
from types import SimpleNamespace

try:
    from src.training.train import load_config, build_dataloaders
    from src.models.q_mmf_model import QuantumMultimodalFramework

    cfg = load_config(PROJECT_ROOT / "experiments/configs/qfl_tensor.yaml")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpts = sorted(CKPT_DIR.glob("*caption*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"Không có checkpoint trong {CKPT_DIR}")

    model = QuantumMultimodalFramework(cfg).to(device)
    state = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(state["model_state_dict"])
    model.eval()
    print(f"Loaded {ckpts[-1].name} (metric={state.get('metric'):.4f})")

    loaders = build_dataloaders(cfg, "captioning")
    batch = next(iter(loaders["test"]))
    with torch.no_grad():
        generated = model(task="caption", images=batch["image"].to(device))

    # Decode token ids -> text
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("\n--- Generated vs Reference ---")
    for i in range(min(5, generated.size(0))):
        hyp_ids = generated[i].tolist()
        ref_ids = [t for t in batch["labels"][i].tolist() if t != -100]
        hyp = tok.decode(hyp_ids, skip_special_tokens=True)
        ref = tok.decode(ref_ids, skip_special_tokens=True)
        print(f"[{i}] HYP: {hyp}\n    REF: {ref}\n")
except FileNotFoundError as e:
    print("Bỏ qua generate (cần checkpoint):", e)

## 4. Export LaTeX table

In [ ]:
if len(df):
    lines = [
        "\\begin{table}[t]",
        "\\centering",
        "\\begin{tabular}{lcc}",
        "\\toprule",
        "Model & BLEU-4 & \\#Params \\\\",
        "\\midrule",
    ]
    for n, r in df.iterrows():
        bleu = f"{r['test_bleu4']:.4f}" if pd.notna(r["test_bleu4"]) else "--"
        params = f"{int(r['total_params']):,}" if pd.notna(r["total_params"]) else "--"
        lines.append(f"{n.replace('_', '\\_')} & {bleu} & {params} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}",
              "\\caption{Image captioning results on Flickr8k test set.}",
              "\\label{tab:caption_results}", "\\end{table}"]
    out = FIG_DIR / "caption_table.tex"
    out.write_text("\n".join(lines))
    print("Saved:", out)